# EDA (Khai phá Dữ liệu): RT-IoT2022 Dataset
Tập dữ liệu RT-IoT2022 từ thư viện UCI Machine Learning Repository (ID: 942) liên quan đến bảo mật và phát hiện xâm nhập mạng IoT.

In [ ]:
# Cài đặt thư viện 

# Import các thư viện cần dùng
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Ẩn các cảnh báo không cần thiết khi chạy notebook
warnings.filterwarnings('ignore')

# Thiết lập giao diện biểu đồ
sns.set_theme(style="whitegrid")

## 1. Tải và Đọc Dữ liệu (Load Data)

Đọc trực tiếp file `RT_IOT2022` trong thư mục `data` của dự án.

In [ ]:
# Import công cụ xử lý đường dẫn
from pathlib import Path

# Tìm file dữ liệu theo cấu trúc thư mục của dự án
cac_duong_dan = [
    Path("../data/RT_IOT2022"),
    Path("data/RT_IOT2022"),
    Path("RT_IOT2022")
]

duong_dan_du_lieu = None

for duong_dan in cac_duong_dan:
    if duong_dan.exists():
        duong_dan_du_lieu = duong_dan
        break

# Kiểm tra file có tồn tại hay không
if duong_dan_du_lieu is None:
    raise FileNotFoundError(
        "Không tìm thấy file RT_IOT2022. "
        "Hãy kiểm tra lại thư mục data."
    )

# Đọc dữ liệu từ file CSV
df = pd.read_csv(duong_dan_du_lieu)

# Bỏ cột đánh số dòng nếu file có cột này
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Xác định cột nhãn cần dự đoán
target_col = "Attack_type"

# Tách dữ liệu đầu vào (X) và nhãn (y)
X = df.drop(columns=[target_col])
y = df[target_col]

print("Đã đọc dữ liệu thành công!")
print("Đường dẫn dữ liệu:", duong_dan_du_lieu)
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])

## 2. Thông tin Tổng quan (Data Overview)
Kiểm tra số lượng mẫu, số lượng đặc trưng và kiểu dữ liệu của từng cột.

In [ ]:
# Hiển thị thông tin tổng quan của DataFrame
df.info()

In [ ]:
# Thống kê mô tả các cột dạng số
# Bao gồm: số lượng, trung bình, độ lệch chuẩn, nhỏ nhất, lớn nhất và các tứ phân vị
df.describe()

In [ ]:
# Hiển thị 5 dòng dữ liệu đầu tiên
df.head()

## 3. Kiểm tra Dữ liệu Thiếu và Trùng lặp (Missing & Duplicates)

In [ ]:
# Đếm số giá trị bị thiếu ở từng cột
print("Số lượng giá trị thiếu theo từng cột:")
print(df.isnull().sum())

# Tính tổng số giá trị thiếu trong toàn bộ dữ liệu
print("\nTổng số giá trị thiếu:", df.isnull().sum().sum())

# Đếm số dòng bị trùng lặp hoàn toàn
print("Số dòng trùng lặp:", df.duplicated().sum())

## 4. Phân tích Biến Mục Tiêu (Target Variable)
Trực quan hóa phân phối của nhãn (ví dụ: các loại tấn công mạng và kết nối bình thường).

In [ ]:
# Đếm số lượng mẫu của từng loại kết nối / tấn công
so_luong_nhan = df[target_col].value_counts()

# In số lượng của từng lớp
print("Số lượng mẫu của từng loại:")
print(so_luong_nhan)

# Vẽ biểu đồ phân phối nhãn
plt.figure(figsize=(10, 6))
sns.countplot(
    data=df,
    y=target_col,
    order=so_luong_nhan.index,
    color="cornflowerblue"
)

plt.title("Phân phối các lớp trong biến Attack_type")
plt.xlabel("Số lượng mẫu")
plt.ylabel("Loại kết nối / Loại tấn công")
plt.tight_layout()
plt.show()

### 4.1. Phân tích các đặc trưng dạng phân loại

Khảo sát số lượng giao thức mạng (`proto`) và dịch vụ (`service`) xuất hiện trong tập dữ liệu.

In [ ]:
# Hiển thị số lượng của từng giao thức mạng
print("Số lượng theo giao thức (proto):")
print(df["proto"].value_counts())

# Vẽ biểu đồ 10 giao thức xuất hiện nhiều nhất
plt.figure(figsize=(8, 5))
sns.countplot(
    data=df,
    y="proto",
    order=df["proto"].value_counts().head(10).index,
    color="orange"
)
plt.title("10 giao thức mạng xuất hiện nhiều nhất")
plt.xlabel("Số lượng")
plt.ylabel("Giao thức")
plt.tight_layout()
plt.show()

In [ ]:
# Hiển thị số lượng của từng dịch vụ
print("Số lượng theo dịch vụ (service):")
print(df["service"].value_counts())

# Vẽ biểu đồ các dịch vụ xuất hiện nhiều nhất
plt.figure(figsize=(8, 5))
sns.countplot(
    data=df,
    y="service",
    order=df["service"].value_counts().head(10).index,
    color="mediumpurple"
)
plt.title("10 dịch vụ xuất hiện nhiều nhất")
plt.xlabel("Số lượng")
plt.ylabel("Dịch vụ")
plt.tight_layout()
plt.show()

## 5. Phân tích Tương quan (Correlation Matrix)
Kiểm tra mức độ tương quan giữa các đặc trưng dạng số (numerical features).

In [ ]:
# Lấy danh sách các cột dữ liệu dạng số
num_cols = df.select_dtypes(include=[np.number]).columns

# Chọn 15 đặc trưng số đầu tiên để biểu đồ dễ quan sát
# Không đưa toàn bộ 80+ cột vào heatmap vì biểu đồ sẽ quá lớn
corr_cols = list(num_cols[:15])

# Tính ma trận tương quan giữa các đặc trưng đã chọn
corr = df[corr_cols].corr()

# Vẽ biểu đồ nhiệt thể hiện mức độ tương quan
plt.figure(figsize=(12, 9))
sns.heatmap(
    corr,
    cmap="coolwarm",
    annot=False,
    linewidths=0.5,
    center=0
)

plt.title("Ma trận tương quan của 15 đặc trưng số đầu tiên")
plt.tight_layout()
plt.show()

## 6. Phân phối các Đặc trưng (Feature Distributions & Outliers)
Sử dụng Boxplot để phát hiện điểm bất thường (outliers) của một vài biến đặc trưng quan trọng đầu tiên.

In [ ]:
# Chọn 4 đặc trưng số tiêu biểu để kiểm tra phân phối và giá trị ngoại lệ
features = [
    "flow_duration",
    "fwd_pkts_tot",
    "bwd_pkts_tot",
    "flow_pkts_per_sec"
]

# Vẽ boxplot cho từng đặc trưng
plt.figure(figsize=(12, 8))

for i, col in enumerate(features, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(y=df[col], color="lightgreen")
    plt.title("Boxplot của " + col)
    plt.ylabel(col)

plt.tight_layout()
plt.show()